# Lekse 17 - Lage lokale AI-agenter med Foundry Local og Qwen

I denne notatboken bygger du en **lokal ingeniørassistent** som kjører helt på din egen arbeidsstasjon. Ingen skybasert inferens brukes på noe tidspunkt. Assistenten kan:

1. **Kalle på verktøy** via Qwen-funksjonskall gjennom Foundry Local.
2. **Liste opp og lese filer** innenfor en sandkasseprosjektmappe.
3. **Analysere kode** med enkle metrikker.
4. **Søke i dokumentasjon** med lokal RAG (Chroma).
5. **Bruke en lokal MCP-server** (går over uten problemer hvis ingen er konfigurert).

Agentkoden ser nesten identisk ut som i sky-leksjonene — det eneste som endres er at klientendepunktet flyttes fra skyen til `localhost`.


## Oppsett

Før du kjører denne notatblokken:

1. **Installer Microsoft Foundry Local** (se [dokumentasjonen](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) for ditt OS).
2. **Last ned og start en Qwen-modell:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Installer Python-pakkene nedenfor.

Alt kjører lokalt. En maskin med ~8 GB RAM er et realistisk minimum.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Koble til Foundry Local

`FoundryLocalManager` laster ned modellen om nødvendig, starter den lokale tjenesten, og gir oss et **OpenAI-kompatibelt endepunkt**. Vi peker deretter den standard OpenAI SDK-en mot det. API-nøkkelen er en lokal plassholder — ingen skylagringslegitimasjon er involvert.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Lokale Verktøy (Sandboxede Filoperasjoner)

Vi oppretter et lite prøveprosjekt på disken, og definerer deretter verktøy som er avgrenset til rotmappen til prosjektet. Sandbox-sjekken er viktig selv lokalt: et verktøy som leser vilkårlige baner kjører med tillatelsene til brukeren din og kan få tilgang til alt du kan.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. Lokal RAG med Chroma

Vi legger inn et lite sett med dokumentasjonsbiter i en lokal Chroma-samling. Chroma kjører i prosessen og lagrer vektorer på disk — ingen server, ingen sky. Verktøyet `search_docs` henter de mest relevante bitene for en forespørsel.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. Verktøysløkken

Nå registrerer vi verktøyene med modellen ved å bruke OpenAI-verktøyskjemaet og kjører den standard verktøysløkken — modellen forespør et verktøy, vi utfører det lokalt, mater resultatet tilbake, og gjentar til modellen produserer et endelig svar. Qwens pålitelige funksjonsanrop er det som gjør at dette fungerer på enheten.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. Lokal MCP (valgfri)

MCP er en transport, ikke en skytjeneste — en MCP-server kan kjøre som en lokal prosess over `stdio`. Cellen nedenfor viser hvordan du kan koble til en lokal MCP-server hvis du har en konfigurert (for eksempel en filsystemserver). Den hopper over grasiøst når `LOCAL_MCP_COMMAND` ikke er satt, slik at notatboken fortsatt kjører fra start til slutt uten den.

Sikkerhetsmerknad: en lokal MCP-server kjører med brukerens rettigheter. Begrens den til en prosjektmappe og valider utdataene før du handler på dem.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Sammendrag

Du bygde en ingeniørassistent som kjører helt lokalt på maskinen din:

- **Foundry Local** leverte en **Qwen**-modell bak en OpenAI-kompatibel endepunkt — slik at agentkoden matcher sky-leksjonene.
- **Sandboxed verktøy** ga agenten filtilgang og kodeanalyse uten å forlate en prosjektmappe.
- **Chroma** ga **lokal RAG** over dokumentasjon.
- **Lokal MCP** viste hvordan MCP-økosystemet kan gjenbrukes offline.

Ingen sky-inferens ble brukt på noe tidspunkt.

### Utfordring

Utvid den lokale agenten til å:

1. **Arbeide med flere MCP-servere** — koble til en filsystemserver og en Git-server og la agenten velge mellom dem.
2. **Bruke lokal minne** — lagre en kort samtalehistorikk på disk slik at assistenten husker tidligere runder på tvers av notatblokkstart.
3. **Støtte lokal multi-agent orkestrering** — legg til en andre lokal agent (f.eks. en anmelder) og la de to samarbeide om en oppgave.

I neste leksjon vil du lære hvordan du sikrer distribuerte AI-agenter.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
